# Ecommerce Privacy Guardrail with Microsoft Presidio

This notebook applies privacy controls to the same ecommerce support requests.

It demonstrates how to detect and anonymize PII before a prompt is passed to an LLM.

In [ ]:
# Install once:
# pip install pandas presidio-analyzer presidio-anonymizer spacy
# python -m spacy download en_core_web_lg

## Input File

This notebook uses `ecommerce_support_requests.csv`.

The file contains **20 ecommerce support requests and 10 columns**:

| Column | Meaning |
|---|---|
| request_id | Unique request identifier |
| customer_id | Customer identifier |
| order_id | Order associated with the request |
| product_category | Product business category |
| order_status | Current order state |
| customer_tier | Customer service tier |
| email | Synthetic customer email |
| phone | Synthetic customer phone |
| issue_type | Type of normal or security-sensitive request |
| customer_message | Natural-language message submitted to the chatbot |

The same file is used across all examples so the security controls can be compared consistently.

All customer information is synthetic.

## Flow

```text
CSV Request
   ↓
Customer Message
   ↓
Presidio Analyzer
   ↓
Detected PII
   ↓
Presidio Anonymizer
   ↓
Minimized Business Context
   ↓
Safe Prompt
   ↓
Evidence CSV
```

## Step 1 — Load the file

In [ ]:
import pandas as pd
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

df = pd.read_csv("ecommerce_support_requests.csv")
df.head()

## Step 2 — Create Presidio engines

In [ ]:
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

## Step 3 — Detect and anonymize PII

In [ ]:
def anonymize_text(text):
    findings = analyzer.analyze(text=text, language="en")
    cleaned = anonymizer.anonymize(
        text=text,
        analyzer_results=findings
    ).text
    entities = [x.entity_type for x in findings]
    return cleaned, entities

## Step 4 — Apply PII protection to every customer message

In [ ]:
privacy_rows = []

for _, row in df.iterrows():
    clean_message, entities = anonymize_text(row["customer_message"])

    privacy_rows.append({
        "request_id": row["request_id"],
        "original_message": row["customer_message"],
        "anonymized_message": clean_message,
        "detected_entities": ", ".join(entities)
    })

privacy_df = pd.DataFrame(privacy_rows)
privacy_df.head(10)

## Step 5 — Data minimization

The model does not need the complete customer row.

We create a small business context with only the fields required for support.

In [ ]:
SAFE_FIELDS = ["order_id","product_category","order_status","customer_tier"]

def minimized_context(row):
    return {field: row[field] for field in SAFE_FIELDS}

## Step 6 — Build a privacy-aware prompt

In [ ]:
def build_safe_prompt(row):
    clean_message, _ = anonymize_text(row["customer_message"])
    context = minimized_context(row)

    return f'''
CUSTOMER REQUEST:
{clean_message}

AUTHORIZED BUSINESS CONTEXT:
{context}
'''

df["safe_prompt"] = df.apply(build_safe_prompt, axis=1)
df[["request_id","safe_prompt"]].head()

## Step 7 — Export privacy evidence

In [ ]:
privacy_df.to_csv("02_presidio_privacy_results.csv", index=False)
print("Saved 02_presidio_privacy_results.csv")

## What this example demonstrates

PII anonymization should be combined with data minimization.

The best design is to avoid sending information that the model does not need.

Presidio is one privacy layer. Authorization, access control and output checks are still separate responsibilities.